# 02 — Compress

Apply the compression matrix to each M0 anchor (PyTorch-native). int8 carries the pre-registered transformer fallback. Every compressed model must still expose .features().

In [2]:
import os
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
print("repo dir exists:", os.path.isdir(REPO))
print("src dir exists:", os.path.isdir(f"{REPO}/src"))
print("drive mounted:", os.path.ismount("/content/drive") or os.path.isdir("/content/drive/MyDrive"))

repo dir exists: False
src dir exists: False
drive mounted: False


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
print("src dir now exists:", os.path.isdir(f"{REPO}/src"))

Mounted at /content/drive
src dir now exists: True


In [4]:
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds
import torch, pandas as pd, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
      "| split:", CFG['split']['primary'])

GPU: Tesla T4 | split: temporal_within_capture


In [5]:
from src import data
df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
print("rows:", len(df), "| splits:", len(splits["train"]), len(splits["val"]), len(splits["test"]))

rows: 3661696 | splits: 2563172 549253 549271


In [7]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
# append a robust loader to train.py (idempotent — checks before appending)
src = open(f"{REPO}/src/train.py").read()
if "def load_anchor(" not in src:
    src += r'''

def load_anchor(dataset, arch, compression, seed, arch_kwargs=None):
    """Reload a saved anchor + its label encoder and scaler. Uses weights_only=False
    (our own trusted checkpoints contain numpy scaler stats)."""
    import numpy as np
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    from . import models as _m
    ckpt = torch.load(PATHS.model(dataset, arch, compression, seed),
                      map_location=DEVICE, weights_only=False)
    model = _m.build(arch, len(ckpt["feat_cols"]), len(ckpt["classes"]),
                     **(arch_kwargs or {})).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"]); model.eval()
    le = LabelEncoder().fit(np.array(ckpt["classes"]))
    scaler = StandardScaler()
    scaler.mean_ = np.asarray(ckpt["scaler_mean"]); scaler.scale_ = np.asarray(ckpt["scaler_scale"])
    scaler.n_features_in_ = len(ckpt["feat_cols"])
    return model, le, scaler, ckpt["feat_cols"]
'''
    open(f"{REPO}/src/train.py","w").write(src)
    print("added train.load_anchor()")
else:
    print("train.load_anchor() already present")

added train.load_anchor()


In [9]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
src = open(f"{REPO}/src/compression.py").read()
print("has build_matrix:", "def build_matrix(" in src)
print("file size:", len(src), "chars")

has build_matrix: False
file size: 2832 chars


In [10]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/compression.py — the compression matrix, PyTorch-native.

Matrix: M0, prune50, prune80, distillation, int8, float16.
Two FAMILIES, and the split is load-bearing for the EXPLAIN stage:
  * post-training, NO retrain: float16, int8  (pure information loss)
  * retrain/finetune-based:    prune50, prune80, distillation  (loss + re-optimisation)
Comparing families is how we dissociate "information lost" from "boundary re-fit".

Every returned object exposes .features() and .forward() so the probe / KernelSHAP /
geometry code reads compressed models identically to the M0 anchor.

int8 carries the PRE-REGISTERED fallback: dynamic quantisation targets Linear layers;
on architectures where it won't yield a clean attribution-accessible model we report
the cell as a documented limitation rather than contorting the pipeline.
"""
from __future__ import annotations
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, TensorDataset

from .config import CFG, PATHS
from . import models as _models
from . import train as _train

DEVICE = _train.DEVICE


# ----------------------------------------------------------------------
# post-training family (no retrain)
# ----------------------------------------------------------------------
def to_float16(model: nn.Module) -> nn.Module:
    """Half precision. Forward expects .half() inputs (the eval helper handles it)."""
    return copy.deepcopy(model).half()


def to_int8(model: nn.Module, arch: str):
    """Dynamic int8 quantisation of Linear layers (post-training, no retrain).
    Returns (quantised_model_or_None, note). Dynamic PTQ runs on CPU.
    Conv1d is not dynamically quantisable; for CNN/transformer this quantises the
    Linear head (+ any Linear blocks) and is reported honestly as partial."""
    m = copy.deepcopy(model).cpu().eval()
    try:
        q = torch.ao.quantization.quantize_dynamic(m, {nn.Linear}, dtype=torch.qint8)
        n_lin = sum(isinstance(mod, nn.Linear) for mod in model.modules())
        note = f"int8 dynamic on {n_lin} Linear layer(s); conv/other layers fp32 (partial, reported)"
        return q, note
    except Exception as e:
        return None, f"int8 skipped for {arch}: {type(e).__name__} {e}"


# ----------------------------------------------------------------------
# retrain family (prune-then-finetune, distillation)
# ----------------------------------------------------------------------
def _magnitude_prune(model: nn.Module, amount: float) -> nn.Module:
    """L1 unstructured prune to `amount` sparsity, made permanent. Class-BLIND by
    design — it deletes low-magnitude weights regardless of which class they serve,
    which is itself a Stage-2 mechanism (does it preferentially hurt rare classes?)."""
    m = copy.deepcopy(model)
    for module in m.modules():
        if isinstance(module, (nn.Linear, nn.Conv1d)):
            prune.l1_unstructured(module, name="weight", amount=amount)
            prune.remove(module, "weight")
    return m


def _reapply_sparsity_mask(model: nn.Module):
    """After fine-tuning a pruned model, zero out the weights that were pruned so
    the sparsity LEVEL is preserved (fine-tune updates would otherwise refill zeros).
    Returns a hook list; call .remove() to detach."""
    masks, hooks = {}, []
    for module in model.modules():
        if isinstance(module, (nn.Linear, nn.Conv1d)):
            masks[module] = (module.weight != 0).float()
    def make_hook(mask):
        def hook(grad):
            return grad * mask
        return hook
    for module, mask in masks.items():
        hooks.append(module.weight.register_hook(make_hook(mask)))
    return hooks, masks


def prune_and_finetune(anchor, df, dataset, splits, seed, amount, *,
                       ft_epochs=8, batch_size=4096, lr=5e-4, arch="cnn1d",
                       verbose=False):
    """Prune the anchor to `amount` sparsity then fine-tune (gradient-masked so the
    sparsity level holds). Returns the fine-tuned pruned model on DEVICE."""
    _train.set_all_seeds(seed)
    feat_cols = _train.feature_columns(df)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df["label"].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits["train"], feat_cols].to_numpy(np.float32))
    t = _train.make_tensors(df, splits, feat_cols, le, scaler)
    Xtr, ytr = t["train"]

    model = _magnitude_prune(anchor, amount).to(DEVICE)
    hooks, _ = _reapply_sparsity_mask(model)        # keep pruned weights at zero during FT
    w = _train.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose:
            print(f"    prune{int(amount*100)} ft epoch {ep}")
    for h in hooks:
        h.remove()
    # final hard mask so saved sparsity is exact
    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, (nn.Linear, nn.Conv1d)):
                module.weight.mul_((module.weight != 0).float())
    return model, le, scaler


def distill(df, dataset, splits, seed, teacher, *, student_kwargs=None,
            T=3.0, alpha=0.5, epochs=12, batch_size=4096, lr=1e-3, arch="cnn1d",
            verbose=False):
    """Train a SMALLER student against the teacher's softened logits (+ true labels).
    student_kwargs sizes the student down from the teacher. Retrain family."""
    _train.set_all_seeds(seed)
    feat_cols = _train.feature_columns(df)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df["label"].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits["train"], feat_cols].to_numpy(np.float32))
    t = _train.make_tensors(df, splits, feat_cols, le, scaler)
    Xtr, ytr = t["train"]

    student = _models.build(arch, len(feat_cols), len(le.classes_),
                            **(student_kwargs or {"channels": (32, 64)})).to(DEVICE)
    teacher = teacher.to(DEVICE).eval()
    w = _train.tempered_class_weights(ytr.numpy(), len(le.classes_))
    hard = nn.CrossEntropyLoss(weight=w)
    kl = nn.KLDivLoss(reduction="batchmean")
    opt = torch.optim.Adam(student.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(epochs):
        student.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with torch.no_grad():
                tlog = teacher(xb)
            slog = student(xb)
            soft = kl(torch.log_softmax(slog / T, 1), torch.softmax(tlog / T, 1)) * (T * T)
            loss = alpha * soft + (1 - alpha) * hard(slog, yb)
            opt.zero_grad(); loss.backward(); opt.step()
        if verbose:
            print(f"    distill epoch {ep}")
    return student, le, scaler


# ----------------------------------------------------------------------
# unified entry point + per-class evaluation across the matrix
# ----------------------------------------------------------------------
def model_size_report(model: nn.Module) -> dict:
    total = sum(p.numel() for p in model.parameters())
    nonzero = sum(int((p != 0).sum()) for p in model.parameters())
    return {"params": total, "nonzero_params": nonzero,
            "sparsity": round(1 - nonzero / max(total, 1), 4)}


def build_matrix(anchor, df, dataset, splits, seed, *, arch="cnn1d",
                 anchor_kwargs=None, cells=None, verbose=True):
    """Produce every compression-matrix cell from the M0 anchor.
    Returns {cell_name: {"model":..., "note":..., "size":..., "is_half":bool, "is_int8":bool}}.
    distillation student defaults to ~1/3 the anchor width."""
    cells = cells or CFG["compression"]["matrix"]
    out = {}
    for cell in cells:
        if verbose: print(f"[{cell}]")
        if cell == "M0":
            m = anchor; note = "uncompressed anchor"; half = i8 = False
        elif cell == "float16":
            m = to_float16(anchor); note = "fp16 .half()"; half, i8 = True, False
        elif cell == "int8":
            m, note = to_int8(anchor, arch); half, i8 = False, (m is not None)
        elif cell.startswith("prune"):
            amt = int(cell.replace("prune", "")) / 100.0
            m, _le, _sc = prune_and_finetune(anchor, df, dataset, splits, seed, amt,
                                             arch=arch, verbose=verbose)
            note = f"L1 prune {int(amt*100)}% + finetune"; half = i8 = False
        elif cell == "distillation":
            sk = {"channels": (24, 48)}   # smaller student than the (64,128) anchor
            m, _le, _sc = distill(df, dataset, splits, seed, anchor,
                                  student_kwargs=sk, arch=arch, verbose=verbose)
            note = "KD student (24,48) <- anchor"; half = i8 = False
        else:
            raise ValueError(cell)
        size = model_size_report(m) if m is not None else {}
        out[cell] = {"model": m, "note": note, "size": size, "is_half": half, "is_int8": i8}
        if verbose and m is not None:
            print(f"   {note} | {size}")
    return out


@torch.no_grad()
def evaluate_cell(entry, df, splits, le, scaler, feat_cols, which="test"):
    """Per-class recall + overall macro-F1 for one compression cell, handling
    fp16/int8 device quirks. Returns (per_class_recall_dict, macro_f1, probs, y_true, y_pred)."""
    from sklearn.metrics import f1_score
    m = entry["model"]
    if m is None:
        return None, None, None, None, None
    sub = df.loc[splits[which]]
    X = scaler.transform(sub[feat_cols].to_numpy(np.float32))
    y_true = le.transform(sub["label"].to_numpy())

    if entry["is_int8"]:
        m = m.cpu().eval(); Xt = torch.tensor(X, dtype=torch.float32)
        logits = torch.cat([m(Xt[i:i+8192]) for i in range(0, len(Xt), 8192)], 0)
    elif entry["is_half"]:
        m = m.to(DEVICE).eval(); Xt = torch.tensor(X, dtype=torch.float16)
        logits = torch.cat([m(Xt[i:i+8192].to(DEVICE)).float().cpu()
                            for i in range(0, len(Xt), 8192)], 0)
    else:
        m = m.to(DEVICE).eval(); Xt = torch.tensor(X, dtype=torch.float32)
        logits = _train._forward_batched(m, Xt)

    probs = torch.softmax(logits, 1).numpy()
    y_pred = probs.argmax(1)
    from .metrics import per_class_recall
    rec = per_class_recall(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro")
    return rec, float(macro), probs, y_true, y_pred
'''
open(f"{REPO}/src/compression.py","w").write(SRC)
print("wrote src/compression.py:", len(SRC), "chars  (expect ~8000+)")
assert "def build_matrix(" in SRC and "def evaluate_cell(" in SRC
print("OK — compression.py written (build_matrix + prune-ft + distill + int8 + eval)")

wrote src/compression.py: 10828 chars  (expect ~8000+)
OK — compression.py written (build_matrix + prune-ft + distill + int8 + eval)


In [11]:
import importlib
from src import data, train, compression, metrics, models
for m in (data, train, compression, metrics): importlib.reload(m)
import pandas as pd, numpy as np, torch

anchor, le, scaler, feat_cols = train.load_anchor("ciciot2023","cnn1d","M0",
                                                  CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
print("anchor reloaded:", anchor.num_params(), "params,", len(le.classes_), "classes")

mat = compression.build_matrix(anchor, df, "ciciot2023", splits, seed=CFG["anchor_seed"],
                               arch="cnn1d", verbose=True)
recs = {}
for cell, entry in mat.items():
    rec, macro, _, _, _ = compression.evaluate_cell(entry, df, splits, le, scaler, feat_cols)
    if rec is None:
        print(f"{cell}: SKIPPED ({entry['note']})"); continue
    recs[cell] = {le.classes_[c]: rec[c] for c in sorted(rec)}
    print(f"{cell:14} macroF1={macro:.4f}  {entry['size'].get('sparsity','')}")
R = pd.DataFrame(recs)
R.to_csv(PATHS.tables("compression","cnn1d_per_class_recall_matrix.csv"))
print("\nsaved per-class recall matrix -> results/tables/compression/")
print("\nPER-CLASS RECALL ACROSS COMPRESSION:")
print(R.round(3).to_string())

anchor reloaded: 29730 params, 34 classes
[M0]
   uncompressed anchor | {'params': 29730, 'nonzero_params': 29730, 'sparsity': 0.0}
[prune50]
    prune50 ft epoch 0
    prune50 ft epoch 1
    prune50 ft epoch 2
    prune50 ft epoch 3
    prune50 ft epoch 4
    prune50 ft epoch 5
    prune50 ft epoch 6
    prune50 ft epoch 7
   L1 prune 50% + finetune | {'params': 29730, 'nonzero_params': 15170, 'sparsity': 0.4897}
[prune80]
    prune80 ft epoch 0
    prune80 ft epoch 1
    prune80 ft epoch 2
    prune80 ft epoch 3
    prune80 ft epoch 4
    prune80 ft epoch 5
    prune80 ft epoch 6
    prune80 ft epoch 7
   L1 prune 80% + finetune | {'params': 29730, 'nonzero_params': 6433, 'sparsity': 0.7836}
[distillation]
    distill epoch 0
    distill epoch 1
    distill epoch 2
    distill epoch 3
    distill epoch 4
    distill epoch 5
    distill epoch 6
    distill epoch 7
    distill epoch 8
    distill epoch 9
    distill epoch 10
    distill epoch 11
   KD student (24,48) <- anchor | {'para

In [12]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "Compression matrix (CNN1D): per-class recall across M0/prune50/prune80/distill/int8/fp16; post-training family preserves trust, prune80 drives class-specific collapse + capacity redistribution"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   2098008..d9c92e7  main -> main



In [13]:
import pandas as pd, numpy as np

# load the two artifacts we already saved
R = pd.read_csv(PATHS.tables("compression","cnn1d_per_class_recall_matrix.csv"), index_col=0)
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)

cells = [c for c in R.columns if c != "M0"]
band = tiers["null_band_2sigma"]          # each class's own 2-sigma noise floor

# Δrecall vs M0, and the same drop expressed in units of that class's band
delta = R[cells].sub(R["M0"], axis=0)
sigma_units = delta.div(band.replace(0, np.nan), axis=0)   # how many 2σ-bands the change is

# a cell is REAL COLLAPSE for a class if it DROPS by more than its 2σ band
collapse_flag = (delta < 0) & (delta.abs() > band.values[:, None])

out = delta.copy()
out["tier"] = tiers["final_tier"]
out["M0"] = R["M0"]
out["band_2s"] = band

# ---- headline: stable_measurable classes only ----
print("="*100)
print("Δ-RECALL vs M0  (negative = recall lost).  *** = real collapse (drop exceeds class 2σ band)")
print("="*100)
meas = out[out["tier"] == "measurable"].sort_values("M0", ascending=False)
for cls, row in meas.iterrows():
    line = f"{cls:24} M0={row['M0']:.3f} band=±{row['band_2s']:.3f} | "
    for c in cells:
        d = row[c]; star = "***" if collapse_flag.loc[cls, c] else "   "
        line += f"{c[:8]:>8}:{d:+.3f}{star} "
    print(line)

# ---- collapse summary per compression cell (measurable tier) ----
print("\n" + "="*60)
print("REAL COLLAPSES per compression cell (stable-measurable tier, n=13):")
meas_flags = collapse_flag.loc[meas.index]
for c in cells:
    hit = meas_flags[c]
    n = int(hit.sum())
    classes = list(meas.index[hit.values])
    print(f"\n  {c}  ({n}/13 classes collapse):")
    for cl in classes:
        print(f"      {cl}: {R.loc[cl,'M0']:.3f} -> {R.loc[cl,c]:.3f}  ({delta.loc[cl,c]:+.3f}, {abs(sigma_units.loc[cl,c]):.1f}× band)")

# save the analysis
delta.to_csv(PATHS.tables("compression","cnn1d_delta_recall.csv"))
collapse_flag.to_csv(PATHS.tables("compression","cnn1d_collapse_flags.csv"))
print("\nsaved Δrecall + collapse flags -> results/tables/compression/")

Δ-RECALL vs M0  (negative = recall lost).  *** = real collapse (drop exceeds class 2σ band)
DoS-HTTP_Flood           M0=0.867 band=±0.038 |  prune50:-0.104     prune80:-0.654*** distilla:-0.025        int8:-0.000     float16:-0.000    
DDoS-HTTP_Flood          M0=0.796 band=±0.068 |  prune50:+0.088     prune80:+0.182    distilla:-0.037        int8:+0.001     float16:+0.001    
DDoS-UDP_Flood           M0=0.742 band=±0.092 |  prune50:-0.134***  prune80:+0.255    distilla:+0.120        int8:+0.001     float16:+0.002    
Recon-HostDiscovery      M0=0.705 band=±0.084 |  prune50:+0.105     prune80:-0.629*** distilla:-0.047***     int8:-0.012***  float16:-0.002    
DoS-UDP_Flood            M0=0.682 band=±0.088 |  prune50:+0.087     prune80:-0.682*** distilla:-0.167***     int8:+0.011     float16:-0.001    
BenignTraffic            M0=0.679 band=±0.075 |  prune50:+0.036     prune80:-0.501*** distilla:+0.038        int8:+0.006     float16:+0.001    
VulnerabilityScan        M0=0.674 band=±0.09

In [14]:
import pandas as pd, numpy as np

R = pd.read_csv(PATHS.tables("compression","cnn1d_per_class_recall_matrix.csv"), index_col=0)
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)
cells = [c for c in R.columns if c != "M0"]
band = tiers["null_band_2sigma"]

delta = R[cells].sub(R["M0"], axis=0)
sigma_units = delta.div(band.replace(0, np.nan), axis=0)     # drop in units of the 2σ band

# CORRECTED: real collapse = a DROP whose magnitude exceeds the FULL 2σ band (sigma_units < -1)
collapse_flag = sigma_units < -1.0

print("="*100)
print("Δ-RECALL vs M0 (negative = recall lost). *** = real collapse (drop > 2σ band, i.e. >1 band-unit)")
print("="*100)
out = delta.copy(); out["tier"] = tiers["final_tier"]; out["M0"] = R["M0"]; out["band"] = band
meas = out[out["tier"]=="measurable"].sort_values("M0", ascending=False)
for cls, row in meas.iterrows():
    line = f"{cls:24} M0={row['M0']:.3f} ±{row['band']:.3f} | "
    for c in cells:
        d = row[c]; star = "***" if collapse_flag.loc[cls,c] else "   "
        line += f"{c[:7]:>7}:{d:+.3f}{star} "
    print(line)

print("\n" + "="*60)
print("REAL COLLAPSES per cell (measurable tier, n=13) — drop exceeds full 2σ band:")
mf = collapse_flag.loc[meas.index]
summary = {}
for c in cells:
    hit = mf[c]; n = int(hit.sum()); summary[c] = n
    print(f"\n  {c}  ({n}/13):")
    for cl in meas.index[hit.values]:
        print(f"      {cl}: {R.loc[cl,'M0']:.3f} -> {R.loc[cl,c]:.3f}  ({delta.loc[cl,c]:+.3f}, {abs(sigma_units.loc[cl,c]):.1f}× band)")

print("\n--- COLLAPSE COUNT BY CELL (the headline number) ---")
for c in cells: print(f"  {c:14}: {summary[c]}/13 classes silently collapse")

delta.to_csv(PATHS.tables("compression","cnn1d_delta_recall.csv"))
collapse_flag.to_csv(PATHS.tables("compression","cnn1d_collapse_flags.csv"))
print("\nsaved corrected Δrecall + collapse flags")

Δ-RECALL vs M0 (negative = recall lost). *** = real collapse (drop > 2σ band, i.e. >1 band-unit)
DoS-HTTP_Flood           M0=0.867 ±0.038 | prune50:-0.104*** prune80:-0.654*** distill:-0.025       int8:-0.000    float16:-0.000    
DDoS-HTTP_Flood          M0=0.796 ±0.068 | prune50:+0.088    prune80:+0.182    distill:-0.037       int8:+0.001    float16:+0.001    
DDoS-UDP_Flood           M0=0.742 ±0.092 | prune50:-0.134*** prune80:+0.255    distill:+0.120       int8:+0.001    float16:+0.002    
Recon-HostDiscovery      M0=0.705 ±0.084 | prune50:+0.105    prune80:-0.629*** distill:-0.047       int8:-0.012    float16:-0.002    
DoS-UDP_Flood            M0=0.682 ±0.088 | prune50:+0.087    prune80:-0.682*** distill:-0.167***    int8:+0.011    float16:-0.001    
BenignTraffic            M0=0.679 ±0.075 | prune50:+0.036    prune80:-0.501*** distill:+0.038       int8:+0.006    float16:+0.001    
VulnerabilityScan        M0=0.674 ±0.092 | prune50:+0.013    prune80:+0.090    distill:-0.091      

In [15]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git add -A
!git commit -q -m "MEASURE headline: band-validated per-class collapse. Post-training (fp16/int8) collapses 0/13 classes; prune80 collapses 10/13 (DoS-HTTP -17.4xband, DoS floods->0). Collapse scales with pruning aggression; family dissociation absolute"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   d9c92e7..afc0add  main -> main



In [1]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/explain.py — Stage 2 (EXPLAIN) mechanism analysis.

Core hypothesis: the classes that collapse under pruning are the ones whose
BASELINE (M0) representation geometry was already most fragile (closest to
Minority Collapse). Pruning, capacity-starved, sacrifices them first.

Two questions, both answered here:
  (a) PREDICTIVE: does M0 per-class geometry predict prune80 Δrecall?
  (b) MECHANISTIC: does pruning push the collapsed classes' geometry further?

Honesty controls (prereg-mandated):
  * geometry is noisy at low class-n -> bootstrap CIs on every per-class measure
  * each rare class paired with a SAME-N frequent-class control, so 'fragile
    geometry' can't be a small-sample artifact
  * features taken from .features() (penultimate), the neural-collapse layer
"""
from __future__ import annotations
import numpy as np
import pandas as pd
import torch

from .config import CFG, PATHS
from . import geometry as geom
from . import train as _train

DEVICE = _train.DEVICE


@torch.no_grad()
def extract_features(model, df, splits, scaler, feat_cols, le, which="test",
                     batch_size=8192, is_half=False):
    """Penultimate representations + logits + labels for a split. Batched (no OOM)."""
    model = model.to(DEVICE).eval()
    sub = df.loc[splits[which]]
    X = scaler.transform(sub[feat_cols].to_numpy(np.float32))
    y = le.transform(sub["label"].to_numpy())
    feats, logits = [], []
    dt = torch.float16 if is_half else torch.float32
    for i in range(0, len(X), batch_size):
        xb = torch.tensor(X[i:i+batch_size], dtype=dt).to(DEVICE)
        feats.append(model.features(xb).float().cpu().numpy())
        logits.append(model(xb).float().cpu().numpy())
    return np.concatenate(feats), np.concatenate(logits), y


def per_class_geometry(feats, logits, y, le, n_boot=200, seed=0, max_per_class=4000):
    """Per-class geometry with bootstrap CIs. Returns a DataFrame indexed by class.
    Columns: mean_cos_to_others (collapse: higher=worse), min_angular_sep_deg,
    etf_gap, margin, plus *_lo/_hi bootstrap bounds. Caps per-class sample for speed."""
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    rows = {}
    etf = geom.etf_deviation(feats, y)
    marg = geom.per_class_margin(logits, y)
    for c in classes:
        rows[int(c)] = {
            "mean_cos_to_others": etf[c]["mean_cos_to_others"],
            "min_angular_sep_deg": etf[c]["min_angular_sep_deg"],
            "etf_gap": etf[c]["etf_gap"],
            "margin": marg[c],
            "support": int((y == c).sum()),
        }
    boot = {int(c): {"mean_cos_to_others": [], "margin": []} for c in classes}
    for b in range(n_boot):
        idx_parts = []
        for c in classes:
            ci = np.where(y == c)[0]
            take = min(len(ci), max_per_class)
            idx_parts.append(rng.choice(ci, take, replace=True))
        bidx = np.concatenate(idx_parts)
        bf, bl, by = feats[bidx], logits[bidx], y[bidx]
        be = geom.etf_deviation(bf, by)
        bm = geom.per_class_margin(bl, by)
        for c in classes:
            boot[int(c)]["mean_cos_to_others"].append(be[c]["mean_cos_to_others"])
            boot[int(c)]["margin"].append(bm[c])
    for c in classes:
        for key in ("mean_cos_to_others", "margin"):
            arr = np.array(boot[int(c)][key])
            rows[int(c)][f"{key}_lo"] = float(np.percentile(arr, 2.5))
            rows[int(c)][f"{key}_hi"] = float(np.percentile(arr, 97.5))
    out = pd.DataFrame(rows).T
    out.index = [le.classes_[int(c)] for c in out.index]
    return out


def same_n_control(feats, y, le, rare_class_name, frequent_class_name, n_boot=200, seed=0):
    """Subsample the FREQUENT class to the rare class's n, recompute its collapse
    geometry. If the rare class's fragility persists vs the same-n frequent control,
    it's real, not a small-sample artifact. Returns (rare_cos, freq_cos_same_n_CI)."""
    rng = np.random.default_rng(seed)
    name_to_idx = {n: i for i, n in enumerate(le.classes_)}
    rc, fc = name_to_idx[rare_class_name], name_to_idx[frequent_class_name]
    n_rare = int((y == rc).sum())
    rare_cos = geom.etf_deviation(feats, y)[rc]["mean_cos_to_others"]
    fi = np.where(y == fc)[0]
    cos_samples = []
    others = feats[y != fc]; others_y = y[y != fc]
    for b in range(n_boot):
        sub = rng.choice(fi, min(n_rare, len(fi)), replace=True)
        ff = np.concatenate([feats[sub], others]); fy = np.concatenate([np.full(len(sub), fc), others_y])
        cos_samples.append(geom.etf_deviation(ff, fy)[fc]["mean_cos_to_others"])
    return rare_cos, (float(np.percentile(cos_samples, 2.5)), float(np.percentile(cos_samples, 97.5)))


def correlate_geometry_vs_collapse(geom_df, delta_recall_series, geom_col="mean_cos_to_others"):
    """The headline test: does baseline geometry predict prune80 collapse?
    Spearman + Kendall (rank, robust) between a geometry measure and Δrecall.
    Positive cos<->negative Δrecall => fragile geometry predicts collapse."""
    from scipy.stats import spearmanr, kendalltau
    common = geom_df.index.intersection(delta_recall_series.index)
    g = geom_df.loc[common, geom_col].astype(float)
    d = delta_recall_series.loc[common].astype(float)
    sr, sp = spearmanr(g, d)
    kt, kp = kendalltau(g, d)
    return {"n": len(common), "spearman_r": float(sr), "spearman_p": float(sp),
            "kendall_tau": float(kt), "kendall_p": float(kp),
            "interpretation": "fragile baseline geometry predicts collapse" if sr < 0 else "no/positive relationship"}
'''
open(f"{REPO}/src/explain.py","w").write(SRC)
print("wrote src/explain.py:", len(SRC), "chars")
assert "per_class_geometry" in SRC and "correlate_geometry_vs_collapse" in SRC and "same_n_control" in SRC
print("OK — explain.py written")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/src/explain.py'

In [8]:
import importlib
from src import data, train, compression, metrics, models
for m in (data, train, compression, metrics): importlib.reload(m)
import pandas as pd, numpy as np, torch

# reload the M0 anchor (robust loader handles the weights_only issue)
anchor, le, scaler, feat_cols = train.load_anchor("ciciot2023","cnn1d","M0",
                                                  CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
print("anchor reloaded:", anchor.num_params(), "params,", len(le.classes_), "classes")

# build + evaluate the matrix
mat = compression.build_matrix(anchor, df, "ciciot2023", splits, seed=CFG["anchor_seed"],
                               arch="cnn1d", verbose=True)
recs = {}
for cell, entry in mat.items():
    rec, macro, _, _, _ = compression.evaluate_cell(entry, df, splits, le, scaler, feat_cols)
    if rec is None:
        print(f"{cell}: SKIPPED ({entry['note']})"); continue
    recs[cell] = {le.classes_[c]: rec[c] for c in sorted(rec)}
    print(f"{cell:14} macroF1={macro:.4f}  {entry['size'].get('sparsity','')}")
R = pd.DataFrame(recs)
R.to_csv(PATHS.tables("compression","cnn1d_per_class_recall_matrix.csv"))
print("\nsaved per-class recall matrix -> results/tables/compression/")
print("\nPER-CLASS RECALL ACROSS COMPRESSION:")
print(R.round(3).to_string())

anchor reloaded: 29730 params, 34 classes


AttributeError: module 'src.compression' has no attribute 'build_matrix'

In [3]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/compression.py — the compression matrix, PyTorch-native.

Matrix: M0, prune50, prune80, distillation, int8, float16.
Two FAMILIES, and the split is load-bearing for the EXPLAIN stage:
  * post-training, NO retrain: float16, int8  (pure information loss)
  * retrain/finetune-based:    prune50, prune80, distillation  (loss + re-optimisation)
Comparing families is how we dissociate "information lost" from "boundary re-fit".

Every returned object exposes .features() and .forward() so the probe / KernelSHAP /
geometry code reads compressed models identically to the M0 anchor.

int8 carries the PRE-REGISTERED fallback: dynamic quantisation targets Linear layers;
on architectures where it won't yield a clean attribution-accessible model we report
the cell as a documented limitation rather than contorting the pipeline.
"""
from __future__ import annotations
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, TensorDataset

from .config import CFG, PATHS
from . import models as _models
from . import train as _train

DEVICE = _train.DEVICE


# ----------------------------------------------------------------------
# post-training family (no retrain)
# ----------------------------------------------------------------------
def to_float16(model: nn.Module) -> nn.Module:
    """Half precision. Forward expects .half() inputs (the eval helper handles it)."""
    return copy.deepcopy(model).half()


def to_int8(model: nn.Module, arch: str):
    """Dynamic int8 quantisation of Linear layers (post-training, no retrain).
    Returns (quantised_model_or_None, note). Dynamic PTQ runs on CPU.
    Conv1d is not dynamically quantisable; for CNN/transformer this quantises the
    Linear head (+ any Linear blocks) and is reported honestly as partial."""
    m = copy.deepcopy(model).cpu().eval()
    try:
        q = torch.ao.quantization.quantize_dynamic(m, {nn.Linear}, dtype=torch.qint8)
        n_lin = sum(isinstance(mod, nn.Linear) for mod in model.modules())
        note = f"int8 dynamic on {n_lin} Linear layer(s); conv/other layers fp32 (partial, reported)"
        return q, note
    except Exception as e:
        return None, f"int8 skipped for {arch}: {type(e).__name__} {e}"


# ----------------------------------------------------------------------
# retrain family (prune-then-finetune, distillation)
# ----------------------------------------------------------------------
def _magnitude_prune(model: nn.Module, amount: float) -> nn.Module:
    """L1 unstructured prune to `amount` sparsity, made permanent. Class-BLIND by
    design — it deletes low-magnitude weights regardless of which class they serve,
    which is itself a Stage-2 mechanism (does it preferentially hurt rare classes?)."""
    m = copy.deepcopy(model)
    for module in m.modules():
        if isinstance(module, (nn.Linear, nn.Conv1d)):
            prune.l1_unstructured(module, name="weight", amount=amount)
            prune.remove(module, "weight")
    return m


def _reapply_sparsity_mask(model: nn.Module):
    """After fine-tuning a pruned model, zero out the weights that were pruned so
    the sparsity LEVEL is preserved (fine-tune updates would otherwise refill zeros).
    Returns a hook list; call .remove() to detach."""
    masks, hooks = {}, []
    for module in model.modules():
        if isinstance(module, (nn.Linear, nn.Conv1d)):
            masks[module] = (module.weight != 0).float()
    def make_hook(mask):
        def hook(grad):
            return grad * mask
        return hook
    for module, mask in masks.items():
        hooks.append(module.weight.register_hook(make_hook(mask)))
    return hooks, masks


def prune_and_finetune(anchor, df, dataset, splits, seed, amount, *,
                       ft_epochs=8, batch_size=4096, lr=5e-4, arch="cnn1d",
                       verbose=False):
    """Prune the anchor to `amount` sparsity then fine-tune (gradient-masked so the
    sparsity level holds). Returns the fine-tuned pruned model on DEVICE."""
    _train.set_all_seeds(seed)
    feat_cols = _train.feature_columns(df)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df["label"].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits["train"], feat_cols].to_numpy(np.float32))
    t = _train.make_tensors(df, splits, feat_cols, le, scaler)
    Xtr, ytr = t["train"]

    model = _magnitude_prune(anchor, amount).to(DEVICE)
    hooks, _ = _reapply_sparsity_mask(model)        # keep pruned weights at zero during FT
    w = _train.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose:
            print(f"    prune{int(amount*100)} ft epoch {ep}")
    for h in hooks:
        h.remove()
    # final hard mask so saved sparsity is exact
    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, (nn.Linear, nn.Conv1d)):
                module.weight.mul_((module.weight != 0).float())
    return model, le, scaler


def distill(df, dataset, splits, seed, teacher, *, student_kwargs=None,
            T=3.0, alpha=0.5, epochs=12, batch_size=4096, lr=1e-3, arch="cnn1d",
            verbose=False):
    """Train a SMALLER student against the teacher's softened logits (+ true labels).
    student_kwargs sizes the student down from the teacher. Retrain family."""
    _train.set_all_seeds(seed)
    feat_cols = _train.feature_columns(df)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df["label"].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits["train"], feat_cols].to_numpy(np.float32))
    t = _train.make_tensors(df, splits, feat_cols, le, scaler)
    Xtr, ytr = t["train"]

    student = _models.build(arch, len(feat_cols), len(le.classes_),
                            **(student_kwargs or {"channels": (32, 64)})).to(DEVICE)
    teacher = teacher.to(DEVICE).eval()
    w = _train.tempered_class_weights(ytr.numpy(), len(le.classes_))
    hard = nn.CrossEntropyLoss(weight=w)
    kl = nn.KLDivLoss(reduction="batchmean")
    opt = torch.optim.Adam(student.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(epochs):
        student.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with torch.no_grad():
                tlog = teacher(xb)
            slog = student(xb)
            soft = kl(torch.log_softmax(slog / T, 1), torch.softmax(tlog / T, 1)) * (T * T)
            loss = alpha * soft + (1 - alpha) * hard(slog, yb)
            opt.zero_grad(); loss.backward(); opt.step()
        if verbose:
            print(f"    distill epoch {ep}")
    return student, le, scaler


# ----------------------------------------------------------------------
# unified entry point + per-class evaluation across the matrix
# ----------------------------------------------------------------------
def model_size_report(model: nn.Module) -> dict:
    total = sum(p.numel() for p in model.parameters())
    nonzero = sum(int((p != 0).sum()) for p in model.parameters())
    return {"params": total, "nonzero_params": nonzero,
            "sparsity": round(1 - nonzero / max(total, 1), 4)}


def build_matrix(anchor, df, dataset, splits, seed, *, arch="cnn1d",
                 anchor_kwargs=None, cells=None, verbose=True):
    """Produce every compression-matrix cell from the M0 anchor.
    Returns {cell_name: {"model":..., "note":..., "size":..., "is_half":bool, "is_int8":bool}}.
    distillation student defaults to ~1/3 the anchor width."""
    cells = cells or CFG["compression"]["matrix"]
    out = {}
    for cell in cells:
        if verbose: print(f"[{cell}]")
        if cell == "M0":
            m = anchor; note = "uncompressed anchor"; half = i8 = False
        elif cell == "float16":
            m = to_float16(anchor); note = "fp16 .half()"; half, i8 = True, False
        elif cell == "int8":
            m, note = to_int8(anchor, arch); half, i8 = False, (m is not None)
        elif cell.startswith("prune"):
            amt = int(cell.replace("prune", "")) / 100.0
            m, _le, _sc = prune_and_finetune(anchor, df, dataset, splits, seed, amt,
                                             arch=arch, verbose=verbose)
            note = f"L1 prune {int(amt*100)}% + finetune"; half = i8 = False
        elif cell == "distillation":
            sk = {"channels": (24, 48)}   # smaller student than the (64,128) anchor
            m, _le, _sc = distill(df, dataset, splits, seed, anchor,
                                  student_kwargs=sk, arch=arch, verbose=verbose)
            note = "KD student (24,48) <- anchor"; half = i8 = False
        else:
            raise ValueError(cell)
        size = model_size_report(m) if m is not None else {}
        out[cell] = {"model": m, "note": note, "size": size, "is_half": half, "is_int8": i8}
        if verbose and m is not None:
            print(f"   {note} | {size}")
    return out


@torch.no_grad()
def evaluate_cell(entry, df, splits, le, scaler, feat_cols, which="test"):
    """Per-class recall + overall macro-F1 for one compression cell, handling
    fp16/int8 device quirks. Returns (per_class_recall_dict, macro_f1, probs, y_true, y_pred)."""
    from sklearn.metrics import f1_score
    m = entry["model"]
    if m is None:
        return None, None, None, None, None
    sub = df.loc[splits[which]]
    X = scaler.transform(sub[feat_cols].to_numpy(np.float32))
    y_true = le.transform(sub["label"].to_numpy())

    if entry["is_int8"]:
        m = m.cpu().eval(); Xt = torch.tensor(X, dtype=torch.float32)
        logits = _train._forward_batched_cpu(m, Xt) if hasattr(_train, "_forward_batched_cpu") \
                 else torch.cat([m(Xt[i:i+8192]) for i in range(0, len(Xt), 8192)], 0)
    elif entry["is_half"]:
        m = m.to(DEVICE).eval(); Xt = torch.tensor(X, dtype=torch.float16)
        logits = torch.cat([m(Xt[i:i+8192].to(DEVICE)).float().cpu()
                            for i in range(0, len(Xt), 8192)], 0)
    else:
        m = m.to(DEVICE).eval(); Xt = torch.tensor(X, dtype=torch.float32)
        logits = _train._forward_batched(m, Xt)

    probs = torch.softmax(logits, 1).numpy()
    y_pred = probs.argmax(1)
    from .metrics import per_class_recall
    rec = per_class_recall(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro")
    return rec, float(macro), probs, y_true, y_pred
'''
open(f"{REPO}/src/compression.py","w").write(SRC)
print("wrote src/compression.py:", len(SRC), "chars")
assert "build_matrix" in SRC and "evaluate_cell" in SRC and "prune_and_finetune" in SRC
print("OK — compression.py written (build_matrix + prune-ft + distill + int8 + eval)")

wrote src/compression.py: 10930 chars
OK — compression.py written (build_matrix + prune-ft + distill + int8 + eval)


In [ ]:
# --- Colab bootstrap (config-driven; never hardcode a path) ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds, require_frozen
set_all_seeds(CFG['anchor_seed'])


## Apply each compression cell

float16 + pruning implemented; distillation + int8 scaffolded.

In [ ]:
from src import compression
# for cell in CFG['compression']['matrix']:
#     mc = compression.apply(m, cell, 'cnn1d', loader=..., calib_loader=..., teacher=m)
#     torch.save(mc, PATHS.model('ciciot2023','cnn1d',cell,CFG['anchor_seed']))

## int8 transformer fallback

If int8 on FT-Transformer won't yield an attribution-accessible model, run int8 on MLP/CNN only and LOG it.

In [ ]:
# document any skipped cell as a limitation, not a silent omission

## End-of-unit (non-negotiable)

In [ ]:
# --- End-of-unit discipline (run before moving on) ---
# 1) outputs saved to Drive  2) commit + push  3) push CSVs/figures  4) confirm sync
# !cd $REPO && git add -A && git commit -m 'nb02: <meaningful message>' && git push
print('Saved under:', PATHS.tables().parent)


In [4]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/explain.py — Stage 2 (EXPLAIN) mechanism analysis.

Core hypothesis: the classes that collapse under pruning are the ones whose
BASELINE (M0) representation geometry was already most fragile (closest to
Minority Collapse). Pruning, capacity-starved, sacrifices them first.

Two questions, both answered here:
  (a) PREDICTIVE: does M0 per-class geometry predict prune80 Δrecall?
  (b) MECHANISTIC: does pruning push the collapsed classes' geometry further?

Honesty controls (prereg-mandated):
  * geometry is noisy at low class-n -> bootstrap CIs on every per-class measure
  * each rare class paired with a SAME-N frequent-class control, so 'fragile
    geometry' can't be a small-sample artifact
  * features taken from .features() (penultimate), the neural-collapse layer
"""
from __future__ import annotations
import numpy as np
import pandas as pd
import torch

from .config import CFG, PATHS
from . import geometry as geom
from . import train as _train

DEVICE = _train.DEVICE


@torch.no_grad()
def extract_features(model, df, splits, scaler, feat_cols, le, which="test",
                     batch_size=8192, is_half=False):
    """Penultimate representations + logits + labels for a split. Batched (no OOM)."""
    model = model.to(DEVICE).eval()
    sub = df.loc[splits[which]]
    X = scaler.transform(sub[feat_cols].to_numpy(np.float32))
    y = le.transform(sub["label"].to_numpy())
    feats, logits = [], []
    dt = torch.float16 if is_half else torch.float32
    for i in range(0, len(X), batch_size):
        xb = torch.tensor(X[i:i+batch_size], dtype=dt).to(DEVICE)
        feats.append(model.features(xb).float().cpu().numpy())
        logits.append(model(xb).float().cpu().numpy())
    return np.concatenate(feats), np.concatenate(logits), y


def per_class_geometry(feats, logits, y, le, n_boot=200, seed=0, max_per_class=4000):
    """Per-class geometry with bootstrap CIs. Returns a DataFrame indexed by class.
    Columns: mean_cos_to_others (collapse: higher=worse), min_angular_sep_deg,
    etf_gap, margin, plus *_lo/_hi bootstrap bounds. Caps per-class sample for speed."""
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    rows = {}
    etf = geom.etf_deviation(feats, y)
    marg = geom.per_class_margin(logits, y)
    for c in classes:
        rows[int(c)] = {
            "mean_cos_to_others": etf[c]["mean_cos_to_others"],
            "min_angular_sep_deg": etf[c]["min_angular_sep_deg"],
            "etf_gap": etf[c]["etf_gap"],
            "margin": marg[c],
            "support": int((y == c).sum()),
        }
    boot = {int(c): {"mean_cos_to_others": [], "margin": []} for c in classes}
    for b in range(n_boot):
        idx_parts = []
        for c in classes:
            ci = np.where(y == c)[0]
            take = min(len(ci), max_per_class)
            idx_parts.append(rng.choice(ci, take, replace=True))
        bidx = np.concatenate(idx_parts)
        bf, bl, by = feats[bidx], logits[bidx], y[bidx]
        be = geom.etf_deviation(bf, by)
        bm = geom.per_class_margin(bl, by)
        for c in classes:
            boot[int(c)]["mean_cos_to_others"].append(be[c]["mean_cos_to_others"])
            boot[int(c)]["margin"].append(bm[c])
    for c in classes:
        for key in ("mean_cos_to_others", "margin"):
            arr = np.array(boot[int(c)][key])
            rows[int(c)][f"{key}_lo"] = float(np.percentile(arr, 2.5))
            rows[int(c)][f"{key}_hi"] = float(np.percentile(arr, 97.5))
    out = pd.DataFrame(rows).T
    out.index = [le.classes_[int(c)] for c in out.index]
    return out


def same_n_control(feats, y, le, rare_class_name, frequent_class_name, n_boot=200, seed=0):
    """Subsample the FREQUENT class to the rare class's n, recompute its collapse
    geometry. If the rare class's fragility persists vs the same-n frequent control,
    it's real, not a small-sample artifact. Returns (rare_cos, freq_cos_same_n_CI)."""
    rng = np.random.default_rng(seed)
    name_to_idx = {n: i for i, n in enumerate(le.classes_)}
    rc, fc = name_to_idx[rare_class_name], name_to_idx[frequent_class_name]
    n_rare = int((y == rc).sum())
    rare_cos = geom.etf_deviation(feats, y)[rc]["mean_cos_to_others"]
    fi = np.where(y == fc)[0]
    cos_samples = []
    others = feats[y != fc]; others_y = y[y != fc]
    for b in range(n_boot):
        sub = rng.choice(fi, min(n_rare, len(fi)), replace=True)
        ff = np.concatenate([feats[sub], others]); fy = np.concatenate([np.full(len(sub), fc), others_y])
        cos_samples.append(geom.etf_deviation(ff, fy)[fc]["mean_cos_to_others"])
    return rare_cos, (float(np.percentile(cos_samples, 2.5)), float(np.percentile(cos_samples, 97.5)))


def correlate_geometry_vs_collapse(geom_df, delta_recall_series, geom_col="mean_cos_to_others"):
    """The headline test: does baseline geometry predict prune80 collapse?
    Spearman + Kendall (rank, robust) between a geometry measure and Δrecall.
    Positive cos<->negative Δrecall => fragile geometry predicts collapse."""
    from scipy.stats import spearmanr, kendalltau
    common = geom_df.index.intersection(delta_recall_series.index)
    g = geom_df.loc[common, geom_col].astype(float)
    d = delta_recall_series.loc[common].astype(float)
    sr, sp = spearmanr(g, d)
    kt, kp = kendalltau(g, d)
    return {"n": len(common), "spearman_r": float(sr), "spearman_p": float(sp),
            "kendall_tau": float(kt), "kendall_p": float(kp),
            "interpretation": "fragile baseline geometry predicts collapse" if sr < 0 else "no/positive relationship"}
'''
open(f"{REPO}/src/explain.py","w").write(SRC)
print("wrote src/explain.py:", len(SRC), "chars")
assert "per_class_geometry" in SRC and "correlate_geometry_vs_collapse" in SRC and "same_n_control" in SRC
print("OK — explain.py written")

wrote src/explain.py: 5573 chars
OK — explain.py written
